In [2]:
# download_all.py
from pathlib import Path
import requests

# ====== Configura tus carpetas base ======
BASE = Path(r"C:\Users\paola\Tesis\01_Data")
DIRS = {
    "RMM": BASE / "RMM",
    "ONI": BASE / "ONI",
    "OLR": BASE / "OLR",
    "SST": BASE / "SST",
}

# ====== Fuentes ======
SOURCES = [
    {
        "name": "RMM (ROMI Wheeler–Hendon)",
        "url": "https://psl.noaa.gov/mjo/mjoindex/romi.cpcolr.1x.txt",
        "dest": DIRS["RMM"] / "romi.cpcolr.1x-RMM.txt",
        "stream": False,  # archivo pequeño
    },
    {
        "name": "ONI (NOAA CPC, HTML tabla)",
        "url": "https://origin.cpc.ncep.noaa.gov/products/analysis_monitoring/ensostuff/ONI_v5.php",
        "dest": DIRS["ONI"] / "ONI_v5.html",
        "stream": False,  # archivo pequeño
    },
    {
        "name": "OLR daily mean (NetCDF)",
        "url": "https://downloads.psl.noaa.gov/Datasets/interp_OLR/olr.day.mean.nc",
        "dest": DIRS["OLR"] / "olr.day.mean.nc",
        "stream": True,   # grande → stream
    },
    {
        "name": "OISST v2 highres (daily mean, NetCDF)",
        "url": "https://downloads.psl.noaa.gov/Datasets/noaa.oisst.v2.highres/sst.day.mean.nc",
        "dest": DIRS["SST"] / "sst.day.mean.nc",
        "stream": True,   # grande → stream
    },
]

def ensure_dirs():
    for p in DIRS.values():
        p.mkdir(parents=True, exist_ok=True)

def download(url: str, dest: Path, stream: bool = False, overwrite: bool = False):
    if dest.exists() and not overwrite:
        print(f"• {dest.name} ya existe, omito (usa overwrite=True si quieres forzar).")
        return
    print(f"↓ Descargando: {url}\n  → {dest}")
    with requests.get(url, stream=stream, timeout=120) as r:
        r.raise_for_status()
        if stream:
            # descarga en chunks para archivos grandes
            with open(dest, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        f.write(chunk)
        else:
            dest.write_bytes(r.content)
    print(f"✓ Listo: {dest}\n")

if __name__ == "__main__":
    ensure_dirs()
    for s in SOURCES:
        try:
            download(s["url"], s["dest"], stream=s["stream"], overwrite=False)
        except Exception as e:
            print(f"⚠️ Error con {s['name']}: {e}\n")
    print("✅ Descargas finalizadas.")


↓ Descargando: https://psl.noaa.gov/mjo/mjoindex/romi.cpcolr.1x.txt
  → C:\Users\paola\Tesis\01_Data\RMM\romi.cpcolr.1x-RMM.txt
✓ Listo: C:\Users\paola\Tesis\01_Data\RMM\romi.cpcolr.1x-RMM.txt

↓ Descargando: https://origin.cpc.ncep.noaa.gov/products/analysis_monitoring/ensostuff/ONI_v5.php
  → C:\Users\paola\Tesis\01_Data\ONI\ONI_v5.html
✓ Listo: C:\Users\paola\Tesis\01_Data\ONI\ONI_v5.html

↓ Descargando: https://downloads.psl.noaa.gov/Datasets/interp_OLR/olr.day.mean.nc
  → C:\Users\paola\Tesis\01_Data\OLR\olr.day.mean.nc
✓ Listo: C:\Users\paola\Tesis\01_Data\OLR\olr.day.mean.nc

↓ Descargando: https://downloads.psl.noaa.gov/Datasets/noaa.oisst.v2.highres/sst.day.mean.nc
  → C:\Users\paola\Tesis\01_Data\SST\sst.day.mean.nc
⚠️ Error con OISST v2 highres (daily mean, NetCDF): 404 Client Error: Not Found for url: https://downloads.psl.noaa.gov/Datasets/noaa.oisst.v2.highres/sst.day.mean.nc

✅ Descargas finalizadas.


In [3]:
pip install requests beautifulsoup4


Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
import re
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://downloads.psl.noaa.gov/Datasets/noaa.oisst.v2.highres/"
DEST_DIR = r"C:\Users\paola\Tesis\01_Data\SST"
os.makedirs(DEST_DIR, exist_ok=True)

# Solo archivos tipo: sst.day.mean.1981.nc, ..., sst.day.mean.2025.nc
FILE_REGEX = re.compile(r"^sst\.day\.mean\.(\d{4})\.nc$")

# Sesión con reintentos básicos
session = requests.Session()
session.headers.update({"User-Agent": "paola-oiSST-downloader/1.0"})
adapter = requests.adapters.HTTPAdapter(max_retries=3)
session.mount("http://", adapter)
session.mount("https://", adapter)

def list_remote_files():
    resp = session.get(BASE_URL, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    files = []
    for a in soup.find_all("a"):
        href = a.get("href", "")
        m = FILE_REGEX.match(href)
        if m:
            year = int(m.group(1))
            files.append((year, href))
    # ordenar por año
    files.sort(key=lambda t: t[0])
    return files

def get_remote_size(url):
    # intenta HEAD para saber tamaño (puede fallar; si falla, retorna None)
    try:
        r = session.head(url, timeout=30, allow_redirects=True)
        if r.status_code == 200 and "Content-Length" in r.headers:
            return int(r.headers["Content-Length"])
    except Exception:
        pass
    return None

def download_with_resume(url, out_path):
    temp_path = out_path + ".part"
    mode = "wb"
    headers = {}
    existing = 0

    remote_size = get_remote_size(url)

    if os.path.exists(out_path):
        # Ya está completo, opcionalmente validar tamaño
        if remote_size is None or os.path.getsize(out_path) == remote_size:
            print(f"✔️ Ya existe: {out_path}")
            return
        else:
            # si difiere, reemplazar
            os.remove(out_path)

    if os.path.exists(temp_path):
        existing = os.path.getsize(temp_path)
        if remote_size and existing >= remote_size:
            # archivo parcial pero del mismo tamaño: renombrar
            os.replace(temp_path, out_path)
            print(f"✔️ Completado (previo): {out_path}")
            return
        if existing > 0:
            headers["Range"] = f"bytes={existing}-"
            mode = "ab"

    with session.get(url, stream=True, timeout=60, headers=headers) as r:
        if r.status_code not in (200, 206):
            r.raise_for_status()
        total = remote_size if remote_size is not None else None
        downloaded = existing
        chunk = 1024 * 1024  # 1 MB
        with open(temp_path, mode) as f:
            for part in r.iter_content(chunk_size=chunk):
                if not part:
                    continue
                f.write(part)
                downloaded += len(part)
                # barra de progreso simple
                if total:
                    pct = downloaded * 100 / total
                    print(f"\r⬇️ {os.path.basename(out_path)}  {pct:5.1f}% ({downloaded/1e6:,.1f} / {total/1e6:,.1f} MB)", end="")
                else:
                    print(f"\r⬇️ {os.path.basename(out_path)}  {downloaded/1e6:,.1f} MB", end="")
    # renombrar al finalizar
    os.replace(temp_path, out_path)
    print(f"\n✅ Descargado: {out_path}")

def main(year_min=None, year_max=None):
    files = list_remote_files()
    if year_min is not None or year_max is not None:
        year_min = year_min if year_min is not None else -10**9
        year_max = year_max if year_max is not None else 10**9
        files = [(y, f) for (y, f) in files if year_min <= y <= year_max]

    if not files:
        print("⚠️ No se encontraron archivos que coincidan.")
        return

    print(f"Se descargarán {len(files)} archivos hacia:\n{DEST_DIR}\n")
    for year, fname in files:
        url = BASE_URL + fname
        out_path = os.path.join(DEST_DIR, fname)
        try:
            download_with_resume(url, out_path)
        except Exception as e:
            print(f"\n❌ Error con {fname}: {e}")

if __name__ == "__main__":
    # Puedes limitar rango con main(1981, 2010). Por defecto descarga todos.
    main()


Se descargarán 45 archivos hacia:
C:\Users\paola\Tesis\01_Data\SST

⬇️ sst.day.mean.1981.nc  100.0% (160.6 / 160.6 MB)
✅ Descargado: C:\Users\paola\Tesis\01_Data\SST\sst.day.mean.1981.nc
⬇️ sst.day.mean.1982.nc  100.0% (482.1 / 482.1 MB)
✅ Descargado: C:\Users\paola\Tesis\01_Data\SST\sst.day.mean.1982.nc
⬇️ sst.day.mean.1983.nc  100.0% (482.5 / 482.5 MB)
✅ Descargado: C:\Users\paola\Tesis\01_Data\SST\sst.day.mean.1983.nc
⬇️ sst.day.mean.1984.nc  100.0% (484.6 / 484.6 MB)
✅ Descargado: C:\Users\paola\Tesis\01_Data\SST\sst.day.mean.1984.nc
⬇️ sst.day.mean.1985.nc  100.0% (482.1 / 482.1 MB)
✅ Descargado: C:\Users\paola\Tesis\01_Data\SST\sst.day.mean.1985.nc
⬇️ sst.day.mean.1986.nc  100.0% (482.0 / 482.0 MB)
✅ Descargado: C:\Users\paola\Tesis\01_Data\SST\sst.day.mean.1986.nc
⬇️ sst.day.mean.1987.nc  100.0% (479.0 / 479.0 MB)
✅ Descargado: C:\Users\paola\Tesis\01_Data\SST\sst.day.mean.1987.nc
⬇️ sst.day.mean.1988.nc  100.0% (477.8 / 477.8 MB)
✅ Descargado: C:\Users\paola\Tesis\01_Data\SST\s